# Demo 07b — Voting, Bagging and Random Forests

**Class 07 · E2, E3** (slides: `slides/07_ensembles.md`)

Why does a combination beat its members, and how do we build members that differ?

* **E2 — Why combining helps, and voting:** the majority of $M$ voters (Condorcet), voters that copy each other, the variance of an average of correlated errors, the identity "error of the mean = mean error minus spread"; then hard, soft and weighted votes of logistic regression, 15-NN and an SVM on the moons.
* **E3 — Bagging and random forests:** the bootstrap sample (63.2 %), bagging in a dozen lines against `BaggingClassifier` and `RandomForestClassifier`, bagged trees with 1, 3, 10 and 100 trees, the out-of-bag error, the correlation of trees against `max_features`, bias and variance of a tree, a bagged tree and a forest over 200 training sets, permutation against impurity importance, and the learners bagging does not help.

Notation of the deck: $S$ members, $f_s$ member $s$, $\rho$ the correlation of two members' errors, $T$ trees. The running example is the moons of Demo 07a (300 points, noise 0.3, 70/30 split). Every cell that prints a number of a slide starts with `slide: <frame title>`.

In [ ]:
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.datasets import make_classification, make_moons
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    RandomForestRegressor,
    VotingClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")

sys.path.append("..")
import ens_utils as eu  # noqa: E402
import plotlib  # noqa: E402

plotlib.setup(module="07")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = eu.BLUE, eu.ORANGE, eu.GREEN, eu.RED, eu.SKY, eu.PURPLE, eu.GREY
SEED = 0
N_JOBS = 8


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line: the frame title of the deck."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    """Print a polars table with rounded floats."""
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))


moons_X, moons_y = make_moons(300, noise=0.3, random_state=SEED)  # the running example of the deck
X_tr, X_te, y_tr, y_te = eu.split(moons_X, moons_y, SEED)  # 210 train, 90 test
LIM = eu.plane_limits(moons_X)
print(f"moons: train {len(y_tr)}, test {len(y_te)}")

## E2 — Why combining helps

**Condorcet.** $M$ voters, each right with probability $p$ and **independent**: the majority is right with the binomial sum $\sum_{k > M/2} \binom{M}{k} p^k (1-p)^{M-k}$. For $M = 3$ and $p = 0.6$: $p^3 + 3p^2(1-p) = 0.216 + 0.432 = 0.648$. With $p = 0.6$ the majority of 101 voters is right 0.979 of the time; with $p = 0.45$ it falls to 0.156, and $p = 0.5$ stays at 0.5. A crowd amplifies whatever its members have.

**The condition is independence.** In the one-factor model a share $c$ of the cases is decided by one answer that all voters copy. At $c = 0.5$ the crowd of 101 stops at 0.790; at $c = 1$ (identical members) it is exactly one voter, 0.600, for any $M$.

In [ ]:
slide("Wisdom of the Crowd: the Majority of M Voters")


def majority_accuracy(M: int, p: float) -> float:
    """P(more than half of M independent voters of accuracy p are right), M odd: a binomial sum."""
    return float(stats.binom.sf(M // 2, M, p))


p3 = 0.6
by_hand = p3**3 + 3 * p3**2 * (1 - p3)
print(
    f"M = 3, p = {p3}: p^3 + 3 p^2 (1 - p) = {p3**3:.3f} + {3 * p3**2 * (1 - p3):.3f} = {by_hand:.3f}; binomial sum {majority_accuracy(3, p3):.3f}"
)
rows = [(M, *[majority_accuracy(M, p) for p in (0.45, 0.5, 0.55, 0.6, 0.7)]) for M in (1, 3, 5, 11, 21, 51, 101)]
show(pl.DataFrame(rows, schema=["M", "p=0.45", "p=0.50", "p=0.55", "p=0.60", "p=0.70"], orient="row"))
print("members with p < 0.5 make the crowd worse: the same sum goes to 0 as M grows; p = 0.5 stays at 0.5")

In [ ]:
slide("Voters That Copy Each Other")


def correlated_majority(M: int, p: float, c: float) -> float:
    """One-factor model: with probability c all M voters copy one common answer (right with probability p),
    otherwise they are independent. Accuracy of the majority = c p + (1 - c) majority_accuracy(M, p)."""
    return c * p + (1 - c) * majority_accuracy(M, p)


rng = np.random.default_rng(SEED)
M_CHECK, N_MC, C_CHECK = 11, 200_000, 0.5
common = rng.random(N_MC) < 0.6
indep = rng.random((N_MC, M_CHECK)) < 0.6
copy = rng.random(N_MC) < C_CHECK
votes = np.where(copy[:, None], common[:, None], indep)
print(
    f"Monte Carlo check, M = {M_CHECK}, p = 0.6, c = {C_CHECK}: simulated {np.mean(votes.sum(axis=1) > M_CHECK / 2):.4f}, formula {correlated_majority(M_CHECK, 0.6, C_CHECK):.4f}"
)
rows = [
    (c, correlated_majority(1, 0.6, c), correlated_majority(11, 0.6, c), correlated_majority(101, 0.6, c))
    for c in (0.0, 0.25, 0.5, 0.75, 1.0)
]
show(pl.DataFrame(rows, schema=["copy share c", "M = 1", "M = 11", "M = 101"], orient="row"))
print("c = 1: identical members, the crowd is one voter and M does not matter")

In [ ]:
# plot e2_condorcet
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.5), sharey=True)
Ms = np.arange(1, 102, 2)
for p, col in ((0.7, GREEN), (0.6, BLUE), (0.55, SKY), (0.5, GREY), (0.45, RED)):
    y_end = [majority_accuracy(int(M), p) for M in Ms]
    axs[0].plot(Ms, y_end, color=col)
    axs[0].text(103, {0.7: 1.04, 0.6: 0.95}.get(p, y_end[-1]), f"$p$ = {p}", color=col, fontsize=6.3, va="center")
for c, col in ((0.0, BLUE), (0.25, SKY), (0.5, ORANGE), (0.75, PURPLE), (1.0, RED)):
    y_end = [correlated_majority(int(M), 0.6, c) for M in Ms]
    axs[1].plot(Ms, y_end, color=col)
    axs[1].text(103, y_end[-1], f"$c$ = {c}", color=col, fontsize=6.3, va="center")
axs[0].set(xlabel="voters $M$, independent", ylabel="accuracy of the majority", ylim=(0, 1.02), xlim=(0, 135))
axs[1].set(xlabel="voters $M$, $p$ = 0.6, share $c$ copy", xlim=(0, 135))
fg.subplots_adjust(wspace=0.08)
display(fg)
plotlib.save(fg, "e2_condorcet")

## The variance of an average of correlated errors

$S$ members with errors of variance $\sigma^2$ and pairwise correlation $\rho$ have an average with variance $\rho\sigma^2 + (1-\rho)\sigma^2 / S$. The simulation (100 000 draws) agrees with the formula to the third decimal. As $S$ grows the second term vanishes and **the floor is $\rho\sigma^2$**: at $\rho = 0.6$ the variance goes from 1.00 to 0.68 with 5 members and stops at 0.61 with 50. Averaging removes the *independent* part of the errors, never the shared part. Bagging works on the first term (more members), a random forest on the floor (a lower $\rho$).

The **ambiguity identity** for squared error holds point by point: $(\bar f - y)^2 = \frac{1}{S}\sum_s (f_s - y)^2 - \frac{1}{S}\sum_s (f_s - \bar f)^2$. On the noisy sine of Demo 07a, five members (polynomial, SVR, MLP, 5-NN, tree) have a mean error of 0.1084; their spread is 0.0153; the mean of the five has an error of 0.0931. The ensemble is **never worse** than the average member, and the gain is exactly the diversity.

In [ ]:
slide("The Variance of an Average of Correlated Errors")
SIGMA = 1.0


def var_of_mean(S: int, rho: float, sigma: float = SIGMA) -> float:
    """Variance of the mean of S errors of variance sigma^2 and pairwise correlation rho: rho s^2 + (1 - rho) s^2 / S."""
    return rho * sigma**2 + (1 - rho) * sigma**2 / S


rng = np.random.default_rng(SEED)
rows = []
for rho in (0.0, 0.3, 0.6, 1.0):
    for S in (1, 5, 50):
        shared = rng.normal(size=(100_000, 1))
        own = rng.normal(size=(100_000, S))
        e = np.sqrt(rho) * shared + np.sqrt(1 - rho) * own  # variance 1, pairwise correlation rho
        rows.append((rho, S, var_of_mean(S, rho), float(e.mean(axis=1).var())))
show(pl.DataFrame(rows, schema=["rho", "S", "formula", "simulated"], orient="row"))
print("S -> infinity: the variance goes to rho sigma^2: more members cannot remove the shared part of the error")

In [ ]:
slide("The Error of the Mean Is Smaller Than the Mean Error")
Xr_tr, yr_tr, Xr_te, yr_te, f_sine = eu.reg_data("sine", SEED)
members = {}
with threadpool_limits(1):
    for name, m in eu.reg_models(SEED).items():
        if name in ("polynomial 5", "SVR RBF", "MLP", "5-NN", "tree"):
            members[name] = m.fit(Xr_tr, yr_tr)
inside = np.abs(Xr_te[:, 0]) <= 3
F = np.array([m.predict(Xr_te[inside]) for m in members.values()])  # S x n
y_in = yr_te[inside]
f_bar = F.mean(axis=0)
mean_member_mse = float(np.mean((F - y_in) ** 2))
ambiguity = float(np.mean((F - f_bar) ** 2))
ens_mse = float(np.mean((f_bar - y_in) ** 2))
print(f"S = {len(members)} members, {inside.sum()} test points in the training range")
print(
    f"mean member MSE {mean_member_mse:.4f} - ambiguity (spread of the members) {ambiguity:.4f} = {mean_member_mse - ambiguity:.4f}; MSE of the mean {ens_mse:.4f}"
)
print(
    f"identity holds point by point: max |difference| = {np.max(np.abs(np.mean((F - y_in) ** 2, axis=0) - np.mean((F - f_bar) ** 2, axis=0) - (f_bar - y_in) ** 2)):.1e}"
)
show(pl.DataFrame({"member": list(members), "test MSE": [float(np.mean((p - y_in) ** 2)) for p in F]}))

In [ ]:
# plot e2_variance
fg, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
Ss = np.arange(1, 51)
for rho, col in ((0.0, BLUE), (0.3, GREEN), (0.6, ORANGE), (1.0, RED)):
    ax.plot(Ss, [var_of_mean(int(S), rho) for S in Ss], color=col, label=f"$\\rho$ = {rho}")
ax.set(xlabel="members $S$", ylabel="variance of the mean / $\\sigma^2$", ylim=(0, 1.05), xlim=(1, 50))
ax.legend(ncols=2, loc="upper right", fontsize=6.3, handlelength=1.2, columnspacing=0.8, labelspacing=0.2)
display(fg)
plotlib.save(fg, "e2_variance")

## Voting on the moons

Three members of three families: logistic regression (Class 03), 15-NN (Class 06), an SVM with an RBF kernel (Class 05). **Hard vote:** each member votes for a class, the majority wins (the panel shows the share of votes). **Soft vote:** the mean of the three predicted probabilities, then 0.5. **Weighted soft vote:** the same with weights equal to the cross-validated accuracy of each member minus 0.5 (the edge over a coin), computed on the training set only.

* On this split (90 test points) the three votes give 0.889, the same as the SVM and below 15-NN (0.956); the soft vote has the better log loss than the hard vote (0.288 against 0.435) because its probabilities are not restricted to $0, 1/3, 2/3, 1$.
* Over **ten seeds** the picture is the same: logistic 0.843, SVM 0.887, 15-NN 0.902 and soft vote 0.887. The vote is **better than the average member and not better than the best one**: the moons with noise 0.3 have a ceiling near 0.92 (Demo 07a), and the best member is already close to it. The next cell shows the same effect for three members of similar quality.

In [ ]:
slide("Voting: Logistic Regression, 15-NN and SVM on the Moons")


class HardVote(BaseEstimator, ClassifierMixin):
    """Majority vote of fitted-on-fit members; `predict_proba` is the share of members that vote for class 1 (weighted)."""

    def __init__(self, members, weights=None):
        self.members, self.weights = members, weights

    def fit(self, X, y):
        self.fitted_ = [clone(m).fit(X, y) for _, m in self.members]
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        w = np.ones(len(self.fitted_)) if self.weights is None else np.asarray(self.weights, dtype=float)
        p1 = (np.array([m.predict(X) for m in self.fitted_]) * w[:, None]).sum(axis=0) / w.sum()
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)


base = eu.gallery_models(SEED)
MEMBERS = [("logistic", base["logistic"]), ("15-NN", base["15-NN"]), ("SVM RBF", base["SVM RBF"])]
cv_acc = [float(cross_val_score(m, X_tr, y_tr, cv=5).mean()) for _, m in MEMBERS]
weights = [a - 0.5 for a in cv_acc]  # the edge over a coin: members that are barely better than chance get little say
print("5-fold CV accuracy on the training set:", ", ".join(f"{n} {a:.3f}" for (n, _), a in zip(MEMBERS, cv_acc)))
print(
    "weights (CV accuracy - 0.5), normalised:",
    ", ".join(f"{n} {w / sum(weights):.3f}" for (n, _), w in zip(MEMBERS, weights)),
)
voters = {
    **{n: clone(m) for n, m in MEMBERS},
    "hard vote": HardVote(MEMBERS),
    "soft vote": VotingClassifier(MEMBERS, voting="soft"),
    "weighted soft": VotingClassifier(MEMBERS, voting="soft", weights=weights),
}
with threadpool_limits(1):
    for m in voters.values():
        m.fit(X_tr, y_tr)
rows = [(n, *eu.scores(m, X_te, y_te)) for n, m in voters.items()]
vote_table = pl.DataFrame(rows, schema=["model", "test acc", "test log loss"], orient="row")
show(vote_table)
d_hs = int(np.sum(voters["hard vote"].predict(X_te) != voters["soft vote"].predict(X_te)))
print(f"hard and soft vote differ on {d_hs} of {len(y_te)} test points")

In [ ]:
# plot e2_voting
fg, axs = eu.new_fig(2, 3, panel_h=1.0)
for ax, (name, m) in zip(axs.ravel(), voters.items()):
    eu.boundary_panel(ax, m, X_tr, y_tr, X_te, y_te, title=name, lim=LIM)
fg.subplots_adjust(wspace=0.06, hspace=0.42)
display(fg)
plotlib.save(fg, "e2_voting")

In [ ]:
slide("Voting: Ten Seeds")


def vote_run(seed):
    X_, y_ = make_moons(300, noise=0.3, random_state=seed)
    a, b, c, d = eu.split(X_, y_, seed)
    g = eu.gallery_models(seed)
    mem = [("logistic", g["logistic"]), ("15-NN", g["15-NN"]), ("SVM RBF", g["SVM RBF"])]
    w = [float(cross_val_score(m, a, c, cv=5).mean()) - 0.5 for _, m in mem]
    ms = {
        **{n: clone(m) for n, m in mem},
        "hard vote": HardVote(mem),
        "soft vote": VotingClassifier(mem, voting="soft"),
        "weighted soft": VotingClassifier(mem, voting="soft", weights=w),
    }
    with threadpool_limits(1):
        return [(n, seed, *eu.scores(m.fit(a, c), b, d)) for n, m in ms.items()]


vr = Parallel(n_jobs=N_JOBS)(delayed(vote_run)(s) for s in range(10))
vl = pl.DataFrame([r for run in vr for r in run], schema=["model", "seed", "acc", "ll"], orient="row")
agg = vl.group_by("model").agg(pl.col("acc").mean(), pl.col("acc").std().alias("acc sd"), pl.col("ll").mean())
order = list(voters)
agg = agg.join(pl.DataFrame({"model": order, "i": range(len(order))}), on="model").sort("i").drop("i")
print("moons, 10 seeds (mean test accuracy, sd over seeds, mean test log loss):")
show(agg)
best_member = max(agg.filter(pl.col("model").is_in(["logistic", "15-NN", "SVM RBF"])).iter_rows(), key=lambda r: r[1])
print(
    f"best member: {best_member[0]} {best_member[1]:.3f}; soft vote {agg.filter(pl.col('model') == 'soft vote')['acc'][0]:.3f}"
)

In [ ]:
slide("Voting: Members of Similar Quality")


def comparable_run(seed):
    X_, y_ = make_moons(300, noise=0.3, random_state=seed)
    a, b, c, d = eu.split(X_, y_, seed)
    g = eu.gallery_models(seed)
    mem = [("SVM RBF", g["SVM RBF"]), ("15-NN", g["15-NN"]), ("tree", g["tree"])]
    with threadpool_limits(1):
        accs = [float(clone(m).fit(a, c).score(b, d)) for _, m in mem]
        hard = float(HardVote(mem).fit(a, c).score(b, d))
        soft = VotingClassifier(mem, voting="soft").fit(a, c)
        return accs + [hard, float(soft.score(b, d)), eu.scores(soft, b, d)[1]]


cr = np.array(Parallel(n_jobs=N_JOBS)(delayed(comparable_run)(s) for s in range(10)))
cn = ["SVM RBF", "15-NN", "tree", "hard vote", "soft vote"]
print("members SVM RBF, 15-NN and a tree (10 seeds, mean test accuracy; sd over seeds):")
show(pl.DataFrame({"model": cn, "acc": cr[:, :5].mean(axis=0), "sd": cr[:, :5].std(axis=0, ddof=1)}))
print(
    f"mean member {cr[:, :3].mean():.3f}; best member (per seed) {cr[:, :3].max(axis=1).mean():.3f}; soft vote {cr[:, 4].mean():.3f}"
)
print(
    f"soft vote against the tree: {int(np.sum(cr[:, 4] > cr[:, 2]))} of 10 seeds better; against the best member: {int(np.sum(cr[:, 4] > cr[:, :3].max(axis=1)))} of 10 seeds better"
)
print(
    "on the moons the ceiling is about 0.92 (see Demo 07a): a vote cannot add much to members that are already near it; it gains over the weakest member and about matches the best"
)

## When a vote fails (no slide)

* **Two weak members outvote the good one.** On XOR logistic regression and naive Bayes are at 0.656 and the SVM at 0.856: the hard vote is 0.656, the soft vote 0.833 (the SVM is more confident, so it can outweigh the two).
* **Identical members** add nothing: three copies of 15-NN vote exactly like one.
* **Members with the same bias** (three linear models) vote linear: 0.756, as each of them.

A vote needs members that are **good** (better than chance), **different** (errors with a low $\rho$) and **independent enough**; the bagging of the next section is a way to produce such members from one learner.

In [ ]:
slide("When a Vote Fails (no slide)")


def fail_case(title, set_name, mem, seed=SEED):
    X_, y_ = eu.class_sets(seed)[set_name]
    a, b, c, d = eu.split(X_, y_, seed)
    with threadpool_limits(1):
        out = [(n, float(clone(m).fit(a, c).score(b, d))) for n, m in mem]
        out.append(("hard vote", float(HardVote(mem).fit(a, c).score(b, d))))
        out.append(("soft vote", float(VotingClassifier(mem, voting="soft").fit(a, c).score(b, d))))
    print(f"{title} ({set_name}, test accuracy):")
    show(pl.DataFrame(out, schema=["model", "test acc"], orient="row"))


g = eu.gallery_models(SEED)
fail_case(
    "two weak members outvote one good member",
    "xor",
    [("logistic", g["logistic"]), ("naive Bayes", g["naive Bayes"]), ("SVM RBF", g["SVM RBF"])],
)
fail_case(
    "identical members: the vote is one model",
    "moons",
    [("15-NN a", g["15-NN"]), ("15-NN b", g["15-NN"]), ("15-NN c", g["15-NN"])],
)
fail_case(
    "members that share the same bias (all linear)",
    "moons",
    [
        ("logistic", g["logistic"]),
        ("naive Bayes", g["naive Bayes"]),
        ("logistic C=100", make_pipeline(StandardScaler(), LogisticRegression(C=100))),
    ],
)

## E3 — Bagging: one learner, many samples

**Bootstrap.** A sample of $n$ draws with replacement from $n$ points misses a given point with probability $(1 - 1/n)^n \to 1/e = 0.368$. A bootstrap sample holds on average **63.2 %** of the distinct points (table: 0.633 for $n = 210$; one sample of $n = 10$ had 5 distinct points), and the other 36.8 % are **out of bag** for that tree.

**Bagging** trains $T$ trees, each on its own bootstrap sample, and averages their probabilities. Written by hand in a dozen lines it reaches 0.933 test accuracy (OOB accuracy 0.933) against 0.878 for one unpruned tree; `BaggingClassifier` gives 0.922 / 0.924 and `RandomForestClassifier` 0.900 / 0.914. They differ in the random streams only; with 90 test points, 0.01 is one point.

With more trees the boundary loses its small boxes but stays a union of axis-parallel boxes (the figure): bagging cuts the **variance** of the trees, it cannot make a diagonal.

The **out-of-bag error** (the vote of the trees that did *not* see each point) is a validation estimate that costs nothing: over 20 seeds it is 0.116 at $T = 100$ against a test error of 0.114, and it stops improving after about 50 trees.

In [ ]:
slide("The Bootstrap Sample: 63.2 % of the Points")
rng = np.random.default_rng(SEED)
rows = []
for n in (10, 50, 210, 1000, 10_000):
    frac = np.mean([len(np.unique(rng.integers(0, n, n))) / n for _ in range(200)])
    rows.append((n, float(frac), 1 - (1 - 1 / n) ** n))
show(pl.DataFrame(rows, schema=["n", "simulated (200 draws)", "1 - (1 - 1/n)^n"], orient="row"), 4)
print(
    f"limit: 1 - 1/e = {1 - math.exp(-1):.4f}; the share of a training set that one tree never sees (out of bag): 1/e = {math.exp(-1):.4f}"
)
idx = rng.integers(0, 10, 10)
print(
    f"one bootstrap sample of n = 10 (indices): {sorted(idx.tolist())}: {len(np.unique(idx))} distinct, {10 - len(np.unique(idx))} left out"
)

In [ ]:
slide("Bagging by Hand")


class HandBag:
    """Bagging of unpruned trees in 12 lines: T bootstrap samples, one tree each, the mean of the T probabilities."""

    def __init__(self, n_trees: int, seed: int = 0, max_features=None):
        self.n_trees, self.seed, self.max_features = n_trees, seed, max_features

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n = len(y)
        self.trees_, self.in_bag_ = [], []
        for t in range(self.n_trees):
            idx = rng.integers(0, n, n)  # a bootstrap sample: n draws with replacement
            tree = DecisionTreeClassifier(max_features=self.max_features, random_state=self.seed * 1000 + t)
            self.trees_.append(tree.fit(X[idx], y[idx]))
            self.in_bag_.append(np.isin(np.arange(n), idx))
        return self

    def predict_proba(self, X):
        p1 = np.mean([t.predict_proba(X)[:, 1] for t in self.trees_], axis=0)
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def oob_error(self, X, y, upto: int | None = None) -> float:
        """Error of the vote of the trees that did not see each point (points that every tree saw are skipped)."""
        T = self.n_trees if upto is None else upto
        votes, counts = np.zeros(len(y)), np.zeros(len(y))
        for t in range(T):
            out = ~self.in_bag_[t]
            votes[out] += self.trees_[t].predict_proba(X[out])[:, 1]
            counts[out] += 1
        seen = counts > 0
        return float(np.mean((votes[seen] / counts[seen] > 0.5) != y[seen]))


hand = HandBag(100, SEED).fit(X_tr, y_tr)
sk_bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED).fit(
    X_tr, y_tr
)
sk_rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=SEED).fit(X_tr, y_tr)
single = DecisionTreeClassifier(random_state=SEED).fit(X_tr, y_tr)
rows = [
    ("one tree", float(single.score(X_te, y_te)), float("nan")),
    ("bagging by hand, 100 trees", float(np.mean(hand.predict(X_te) == y_te)), 1 - hand.oob_error(X_tr, y_tr)),
    ("BaggingClassifier, 100 trees", float(sk_bag.score(X_te, y_te)), float(sk_bag.oob_score_)),
    ("RandomForestClassifier, 100 trees", float(sk_rf.score(X_te, y_te)), float(sk_rf.oob_score_)),
]
show(pl.DataFrame(rows, schema=["model", "test acc", "OOB acc"], orient="row"))
print(
    f"points that every one of the 100 trees saw: {int(np.sum(np.all(hand.in_bag_, axis=0)))} of {len(y_tr)} (all points have an OOB vote)"
)
print(
    "the hand-made bag and the library differ only in the random streams: same method, same accuracy up to the noise of 90 test points"
)

In [ ]:
slide("Bagged Trees: 1, 3, 10 and 100 Trees")
bags = {T: HandBag(T, SEED).fit(X_tr, y_tr) for T in (1, 3, 10, 100)}
rows = [(T, *eu.scores(b, X_te, y_te), 1 - b.oob_error(X_tr, y_tr)) for T, b in bags.items()]
show(pl.DataFrame(rows, schema=["trees T", "test acc", "test log loss", "OOB acc"], orient="row"))

In [ ]:
# plot e3_bagging
fg, axs = eu.new_fig(1, 4, panel_h=1.25)
for ax, (T, b) in zip(axs, bags.items()):
    eu.boundary_panel(ax, b, X_tr, y_tr, X_te, y_te, title=f"{T} tree{'s' if T > 1 else ''}", lim=LIM)
fg.subplots_adjust(wspace=0.06)
display(fg)
plotlib.save(fg, "e3_bagging")

In [ ]:
slide("Out-of-Bag Error Against the Number of Trees")


def oob_curve(seed, T_max=200):
    X_, y_ = make_moons(300, noise=0.3, random_state=seed)
    a, b, c, d = eu.split(X_, y_, seed)
    with threadpool_limits(1):
        bag = HandBag(T_max, seed).fit(a, c)
        grid = [1, 2, 3, 5, 10, 20, 30, 50, 100, 150, 200]
        probs = np.cumsum([t.predict_proba(b)[:, 1] for t in bag.trees_], axis=0)
        return [(T, bag.oob_error(a, c, T), float(np.mean(((probs[T - 1] / T) > 0.5) != d))) for T in grid]


oc = Parallel(n_jobs=N_JOBS)(delayed(oob_curve)(s) for s in range(20))
ocl = pl.DataFrame(
    [(*r, s) for s, run in enumerate(oc) for r in run], schema=["T", "oob", "test", "seed"], orient="row"
)
oca = (
    ocl.group_by("T")
    .agg(
        pl.col("oob").mean(),
        pl.col("oob").std().alias("oob sd"),
        pl.col("test").mean(),
        pl.col("test").std().alias("test sd"),
    )
    .sort("T")
)
show(oca)
t1, t100 = oca.filter(pl.col("T") == 1), oca.filter(pl.col("T") == 100)
print(
    f"OOB error: {t1['oob'][0]:.3f} with 1 tree, {t100['oob'][0]:.3f} with 100 trees; test error {t1['test'][0]:.3f} and {t100['test'][0]:.3f} (mean of 20 seeds)"
)
print(
    f"gap between the OOB error and the test error at T = 100: {abs(t100['oob'][0] - t100['test'][0]):.3f}: the OOB error is a free validation estimate"
)

In [ ]:
# plot e3_oob
fg, ax = plotlib.fig("col", 1.5)
for col, key, label, ls in ((BLUE, "oob", "out-of-bag error", "-"), (ORANGE, "test", "test error", "--")):
    m, s = oca[key].to_numpy(), oca[key + " sd"].to_numpy()
    ax.plot(oca["T"], m, color=col, ls=ls, marker="o", ms=2.5, label=label)
    ax.fill_between(oca["T"], m - s, m + s, color=col, alpha=0.15, linewidth=0)
ax.set(xscale="log", xlabel="trees $T$ (log scale)", ylabel="error", ylim=(0.05, 0.3))
ax.legend(loc="upper right", fontsize=6.3, handlelength=1.4)
display(fg)
plotlib.save(fg, "e3_oob")

## A random forest decorrelates the trees

Bagged trees share their best splits (the same strong features), so their errors are correlated; a **random forest** lets each split choose among $m$ = `max_features` features drawn at random. On a 20-feature set (6 informative, 4 redundant) five seeds give: the correlation $\rho$ of two trees' errors rises from 0.11 ($m = 1$) to 0.28 ($m = 20$, plain bagging); a single tree gets *worse* when $m$ is small (error 0.33 at $m = 1$ against 0.23 at $m = 20$). The forest error is lowest at $m = 10$ (0.116), 0.127 at the default $m = \sqrt{d} = 4$ and 0.127 for bagging: a **trade-off between strength and correlation**, and a flat optimum, which is why `max_features` is the first hyper-parameter to tune.

In [ ]:
slide("A Random Forest Decorrelates the Trees: max_features")
D_FEAT = 20


def rho_run(seed, max_features):
    X_, y_ = make_classification(
        1000,
        n_features=D_FEAT,
        n_informative=6,
        n_redundant=4,
        n_clusters_per_class=2,
        flip_y=0.05,
        class_sep=1.0,
        random_state=seed,
    )
    a, b, c, d = eu.split(X_, y_, seed)
    with threadpool_limits(1):
        rf = RandomForestClassifier(n_estimators=100, max_features=max_features, oob_score=True, random_state=seed).fit(
            a, c
        )
    tree_pred = np.array([t.predict(b.astype(np.float32)) for t in rf.estimators_])
    err = (tree_pred != d).astype(float)
    return (
        max_features,
        float(eu.mean_pairwise_corr(err)),
        float(err.mean()),
        float(np.mean(rf.predict(b) != d)),
        1 - float(rf.oob_score_),
    )


FEATS = [1, 2, 4, 6, 10, 15, 20]
rr = Parallel(n_jobs=N_JOBS)(delayed(rho_run)(s, mf) for s in range(5) for mf in FEATS)
rl = pl.DataFrame(rr, schema=["max_features", "rho", "tree error", "forest error", "oob error"], orient="row")
rho_t = rl.group_by("max_features").agg(pl.all().mean()).sort("max_features")
show(rho_t)
print(
    f"d = {D_FEAT} features, 1000 points, 5 seeds; rho = mean pairwise correlation of the 0/1 errors of the 100 trees on the test set"
)
print("max_features = 20 is plain bagging; sqrt(d) = 4 is the default of scikit-learn for classification")

In [ ]:
# plot e3_maxfeat
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.45))
axs[0].plot(rho_t["max_features"], rho_t["rho"], color=BLUE, marker="o", ms=3)
axs[0].set(xlabel="max_features $m$ (of 20)", ylabel="correlation $\\rho$ of two trees' errors", ylim=(0, None))
axs[1].plot(rho_t["max_features"], rho_t["tree error"], color=GREY, marker="s", ms=3, ls="--", label="one tree (mean)")
axs[1].plot(rho_t["max_features"], rho_t["forest error"], color=ORANGE, marker="o", ms=3, label="forest of 100 trees")
axs[1].set(xlabel="max_features $m$ (of 20)", ylabel="test error")
axs[1].legend(loc="upper right", fontsize=6.3, handlelength=1.4)
fg.subplots_adjust(wspace=0.38)
display(fg)
plotlib.save(fg, "e3_maxfeat")

## Bias and variance of a tree, a bagged tree and a forest

200 independent training sets of 300 points of a noisy target (one strong sine feature and six weak features among ten correlated ones, noise sd 0.5, so $\sigma^2 = 0.25$); for each model, bias$^2$ and variance of the prediction on 500 fixed test points, expected test MSE = noise + bias$^2$ + variance (the two columns agree).

* The unpruned **tree** has little bias (0.105) and a huge variance (1.038).
* **Bagging** cuts the variance by a factor of 7 (0.140) and adds a little bias (0.131): the expected MSE falls from 1.39 to 0.52.
* A **forest** with $m = 5$ lowers the variance again (0.097) and gives the best total (0.472); with $m = 3$ the variance is smaller still (0.086) but the bias grows to 0.170 and the total rises to 0.507: a small $m$ is too strong a restriction.

The 1-D picture shows the same thing: 30 trees fitted to 30 samples of one noisy sine wander around the truth (variance 0.091), 30 bagged fits wander half as much (0.046).

In [ ]:
slide("Bias and Variance of a Tree, a Bagged Tree and a Forest")
N_SETS, N_TRAIN, SD, D_BV, CORR = 200, 300, 0.5, 10, 0.5


def draw_x(rng, n):
    """n points of 10 features with pairwise correlation 0.5 (one shared factor): the correlated features of real tables."""
    return np.sqrt(1 - CORR) * rng.normal(size=(n, D_BV)) + np.sqrt(CORR) * rng.normal(size=(n, 1))


def truth(X):
    """2 sin(x1) + 0.6 (tanh(x2) + ... + tanh(x7)): one strong feature, six weak ones, three pure noise columns."""
    return 2 * np.sin(X[:, 0]) + 0.6 * np.tanh(X[:, 1:7]).sum(axis=1)


X_grid = draw_x(np.random.default_rng(99), 500)
f_grid = truth(X_grid)
FORESTS = {"bagged": 1.0, "forest m=5": 5, "forest m=3": 3}


def bv_run(seed):
    rng = np.random.default_rng(seed)
    Xs = draw_x(rng, N_TRAIN)
    ys = truth(Xs) + rng.normal(0, SD, N_TRAIN)
    with threadpool_limits(1):
        out = {"tree": DecisionTreeRegressor(random_state=seed).fit(Xs, ys).predict(X_grid)}
        for name, mf in FORESTS.items():
            out[name] = (
                RandomForestRegressor(n_estimators=50, max_features=mf, random_state=seed).fit(Xs, ys).predict(X_grid)
            )
    return out


runs = Parallel(n_jobs=N_JOBS)(delayed(bv_run)(s) for s in range(N_SETS))
rows = []
for name in ("tree", *FORESTS):
    P = np.array([r[name] for r in runs])  # sets x points
    bias2 = float(np.mean((P.mean(axis=0) - f_grid) ** 2))
    var = float(np.mean(P.var(axis=0)))
    mse = float(np.mean((P - f_grid) ** 2)) + SD**2
    rows.append((name, bias2, var, SD**2, bias2 + var + SD**2, mse))
bv = pl.DataFrame(rows, schema=["model", "bias^2", "variance", "noise", "sum", "expected test MSE"], orient="row")
show(bv)
print(
    f"{N_SETS} training sets of {N_TRAIN} points, {D_BV} correlated features (rho 0.5, 7 informative), noise sd {SD}; bias^2 and variance are averaged over 500 fixed test points"
)
print(
    "bagged = 50 trees on bootstrap samples with all 10 features per split; forest m = the same with m features drawn at each split"
)
print("the decomposition holds: 'sum' and 'expected test MSE' agree")

In [ ]:
# plot e3_biasvar
fg, ax = plotlib.fig("col", 1.6)
names = bv["model"].to_list()
x = np.arange(len(names))
b2, va, no = bv["bias^2"].to_numpy(), bv["variance"].to_numpy(), bv["noise"].to_numpy()
ax.bar(x, no, color=plotlib.LIGHT, label="noise \u03c3\u00b2", width=0.6)
ax.bar(x, b2, bottom=no, color=BLUE, label="bias\u00b2", width=0.6)
ax.bar(x, va, bottom=no + b2, color=ORANGE, label="variance", width=0.6)
for i in range(len(names)):
    ax.text(x[i], no[i] + b2[i] + va[i] + 0.02, f"{no[i] + b2[i] + va[i]:.2f}", ha="center", va="bottom", fontsize=6.3)
    ax.text(x[i], no[i] + b2[i] + va[i] / 2, f"{va[i]:.2f}", ha="center", va="center", fontsize=5.6)
ax.set(
    xticks=x,
    xticklabels=[n.replace(" ", "\n") for n in names],
    ylabel="expected test MSE",
    ylim=(0, (no + b2 + va).max() * 1.2),
)
ax.grid(axis="x", visible=False)
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, 1.2), fontsize=6.0, handlelength=1.0, columnspacing=0.7)
ax.tick_params(axis="x", labelsize=6.5)
display(fg)
plotlib.save(fg, "e3_biasvar")

In [ ]:
# plot e3_spaghetti: 30 fits of a tree and of a bagged tree on one noisy sine
rng = np.random.default_rng(SEED)
xs = np.linspace(-3, 3, 300)[:, None]
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.35), sharey=True)
fits = {"tree": [], "bagged (25 trees)": []}
for k in range(30):
    x1 = rng.uniform(-3, 3, 60)
    y1 = np.sin(1.5 * x1) + rng.normal(0, 0.3, 60)
    fits["tree"].append(DecisionTreeRegressor(random_state=k).fit(x1[:, None], y1).predict(xs))
    fits["bagged (25 trees)"].append(
        RandomForestRegressor(n_estimators=25, max_features=1.0, random_state=k).fit(x1[:, None], y1).predict(xs)
    )
for ax, (name, P), col in zip(axs, fits.items(), (RED, BLUE)):
    P = np.array(P)
    for p in P:
        ax.plot(xs[:, 0], p, color=col, alpha=0.18, lw=0.7)
    ax.plot(xs[:, 0], np.sin(1.5 * xs[:, 0]), color="k", lw=1.0, ls=":")
    ax.set(xlabel="$x$", title=f"{name}: variance {P.var(axis=0).mean():.3f}", xlim=(-3, 3))
    ax.title.set_fontsize(7)
axs[0].set_ylabel("prediction (30 fits, 60 points each)")
fg.subplots_adjust(wspace=0.08)
display(fg)
plotlib.save(fg, "e3_spaghetti")

## Feature importance: impurity against permutation

The moons plus a noisy copy of $x_1$ and three columns of pure noise (a continuous one, an integer 0-9, a binary one). The **impurity importance** (the total decrease of Gini in the splits of a feature, computed on the training set) gives the random continuous column 0.088, about 43 % of what it gives $x_1$, and the three noise columns together 14.1 %. The **permutation importance** (the drop of test accuracy when one column is shuffled) gives the same column 0.014 and the three together 7.4 %: noise features that a fully grown tree can use to memorise the training set get credit in the first measure and almost none in the second. The copy of $x_1$ shows the other weakness: the signal is **shared** between $x_1$ (0.078) and its copy (0.065), so each looks less important than the information they carry together.

In [ ]:
slide("Feature Importance: Impurity Against Permutation")
rng = np.random.default_rng(SEED)
Xm, ym = make_moons(500, noise=0.3, random_state=SEED)
copy = Xm[:, 0] + rng.normal(0, 0.15, len(ym))
extra = np.c_[copy, rng.normal(size=len(ym)), rng.integers(0, 10, len(ym)), rng.integers(0, 2, len(ym))]
Xi = np.c_[Xm, extra]
FEAT = ["x1", "x2", "x1 copy", "random continuous", "random 0-9", "random 0/1"]
a, b, c, d = eu.split(Xi, ym, SEED)
with threadpool_limits(1):
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(a, c)
    perm = permutation_importance(rf, b, d, n_repeats=30, random_state=SEED)
mdi = rf.feature_importances_
imp = pl.DataFrame(
    {
        "feature": FEAT,
        "impurity (train)": mdi,
        "permutation (test)": perm.importances_mean,
        "perm sd": perm.importances_std,
    }
)
show(imp)
print(
    f"forest test accuracy {rf.score(b, d):.3f}; the three noise columns get {mdi[3:].sum():.3f} of the impurity importance and {perm.importances_mean[3:].sum():.3f} (accuracy drop) of the permutation importance"
)
print(
    f"shares: the three random columns hold {mdi[3:].sum():.1%} of the impurity importance and {perm.importances_mean[3:].sum() / perm.importances_mean.sum():.1%} of the permutation importance"
)
print(
    f"x1 and its noisy copy share the signal: permuting x1 alone costs {perm.importances_mean[0]:.3f}; the copy {perm.importances_mean[2]:.3f}"
)

In [ ]:
# plot e3_importance
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.5), sharey=True)
y_pos = np.arange(len(FEAT))[::-1]
axs[0].barh(y_pos, mdi, color=BLUE, height=0.6)
axs[0].set(xlabel="impurity importance")
axs[1].barh(y_pos, perm.importances_mean, xerr=perm.importances_std, color=ORANGE, height=0.6, error_kw={"lw": 0.6})
axs[1].set(xlabel="accuracy drop when permuted")
axs[0].set_yticks(y_pos, FEAT)
for ax in axs:
    ax.grid(axis="y", visible=False)
fg.subplots_adjust(wspace=0.12)
display(fg)
plotlib.save(fg, "e3_importance")

## Bagging helps unstable learners only

Over ten seeds: a tree gains **0.020** from bagging (9 of 10 seeds better), the stable 15-NN loses 0.006, logistic regression 0.002, and 1-NN gains exactly nothing: its prediction is the label of the nearest neighbour, which a bootstrap sample keeps with probability 0.632, and the vote of the others agrees with it. Bagging reduces **variance**; a learner with little variance (a line, a smooth k-NN) has nothing to give. This is also why bagged *stumps* or bagged *linear* members are a poor design, and why boosting (Demo 07c) uses weak learners in another way.

In [ ]:
slide("Bagging Helps Unstable Learners Only")


def stability_run(seed):
    X_, y_ = make_moons(300, noise=0.3, random_state=seed)
    a, b, c, d = eu.split(X_, y_, seed)
    g = eu.gallery_models(seed)
    ms = {
        "tree": DecisionTreeClassifier(random_state=seed),
        "bagged tree": BaggingClassifier(DecisionTreeClassifier(), n_estimators=50, random_state=seed),
        "15-NN": g["15-NN"],
        "bagged 15-NN": BaggingClassifier(g["15-NN"], n_estimators=50, random_state=seed),
        "1-NN": g["1-NN"],
        "bagged 1-NN": BaggingClassifier(g["1-NN"], n_estimators=50, random_state=seed),
        "logistic": g["logistic"],
        "bagged logistic": BaggingClassifier(g["logistic"], n_estimators=50, random_state=seed),
    }
    with threadpool_limits(1):
        return [(n, seed, float(m.fit(a, c).score(b, d))) for n, m in ms.items()]


sr = Parallel(n_jobs=N_JOBS)(delayed(stability_run)(s) for s in range(10))
sl = pl.DataFrame([r for run in sr for r in run], schema=["model", "seed", "acc"], orient="row")
order = ["tree", "bagged tree", "15-NN", "bagged 15-NN", "1-NN", "bagged 1-NN", "logistic", "bagged logistic"]
sa = (
    sl.group_by("model")
    .agg(pl.col("acc").mean(), pl.col("acc").std().alias("sd"))
    .join(pl.DataFrame({"model": order, "i": range(8)}), on="model")
    .sort("i")
    .drop("i")
)
show(sa)
for single_name in ("tree", "15-NN", "1-NN", "logistic"):
    gain = (
        sl.filter(pl.col("model") == f"bagged {single_name}")["acc"].to_numpy()
        - sl.filter(pl.col("model") == single_name)["acc"].to_numpy()
    )
    print(
        f"bagging {single_name}: mean gain {gain.mean():+.3f} (sd over seeds {gain.std(ddof=1):.3f}, {int(np.sum(gain > 0))} of 10 seeds better)"
    )
print(
    "bagging averages out the variance of a learner that changes with the sample: the tree gains; 1-NN (stable here), 15-NN and logistic regression do not"
)

## Summary

* A majority of independent voters better than chance improves with $M$; correlated errors set a floor $\rho\sigma^2$ (E2).
* A vote of members of similar quality about matches the best one and beats the average one; it fails when the members are bad, identical or share their bias.
* Bagging = bootstrap + average: 63.2 % of the points per tree, an OOB error for free, variance cut by a factor of 7 for a tree (E3).
* A random forest adds `max_features` to lower $\rho$; too small an $m$ weakens the trees.
* Bagging helps unstable learners (trees); it does nothing for stable ones.
* Next: members that are built to fix each other's mistakes (`03_boosting.ipynb`).